# 04 - Classification
## Peak/Off-Peak Demand Classification

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.data.terna_loader import load_demand_data
from src.data.feature_engineering import build_feature_matrix, prepare_train_test, get_feature_columns
from src.models.ml_models import classify_peak_offpeak, train_peak_classifier

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12

## Load and Prepare Data

In [ ]:
demand = load_demand_data()
df = build_feature_matrix(demand, target_col='demand_mw')
df = df.dropna()

df = classify_peak_offpeak(df, target_col='demand_mw')

print(f'Class distribution:')
print(df['is_peak'].value_counts())
print(f'\nPeak ratio: {df["is_peak"].mean():.2%}')

In [ ]:
feature_cols = get_feature_columns(df, target_col='demand_mw')
feature_cols = [c for c in feature_cols if c != 'is_peak']

train, test = prepare_train_test(df, test_ratio=0.2)

X_train = train[feature_cols]
y_train = train['is_peak']
X_test = test[feature_cols]
y_test = test['is_peak']

print(f'Train: {X_train.shape}, Test: {X_test.shape}')

## Train Classifier

In [ ]:
clf_model, clf_pred, clf_metrics = train_peak_classifier(X_train, y_train, X_test, y_test)

print('Classification Report:')
report = clf_metrics['classification_report']
print(pd.DataFrame(report).T)
print(f'\nAUC-ROC: {clf_metrics["auc_roc"]:.4f}')

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ConfusionMatrixDisplay.from_predictions(y_test, clf_pred, ax=axes[0], cmap='Blues')
axes[0].set_title('Confusion Matrix')

from sklearn.metrics import roc_curve
y_prob = clf_model.predict_proba(X_test)[:, 1]
fpr, tpr, _ = roc_curve(y_test, y_prob)
axes[1].plot(fpr, tpr, label=f'AUC = {clf_metrics["auc_roc"]:.4f}')
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.5)
axes[1].set_title('ROC Curve')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].legend()

plt.tight_layout()
plt.show()

## Feature Importance for Classification

In [ ]:
importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': clf_model.feature_importances_,
}).sort_values('importance', ascending=False)

top_n = 15
fig, ax = plt.subplots(figsize=(10, 8))
top_features = importance.head(top_n)
ax.barh(range(top_n), top_features['importance'].values[::-1])
ax.set_yticks(range(top_n))
ax.set_yticklabels(top_features['feature'].values[::-1])
ax.set_title(f'Top {top_n} Features - Peak Classification')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

In [ ]:
print('Notebook 04 complete. Classification model trained.')